# AMANAH v0.2 — CPU-only validated publisher (patched)

This notebook needs **no GPU** and creates **no Hugging Face endpoint, subscription, or credits**. It reuses the already-trained checkpoint and measured reports from Google Drive, but rebuilds the Hugging Face package from the **latest GitHub v0.2 code** before testing and upload. This avoids publishing stale runtime code from the earlier cache.


In [ ]:
# 1) Mount Drive and locate the exact successful run
from google.colab import drive, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys
drive.mount('/content/drive', force_remount=False)
CACHE_ROOT=Path('/content/drive/MyDrive/semantic_integrity_model_cache')
RUN_SIGNATURE='74751fc19ce83158'
RUN_CACHE=CACHE_ROOT/'runs'/RUN_SIGNATURE
checkpoint=RUN_CACHE/'checkpoint'
cached_model_repository=RUN_CACHE/'model_repository'
validation_report=RUN_CACHE/'VALIDATION_REPORT.json'
acceptance_report=RUN_CACHE/'FINAL_ACCEPTANCE_REPORT.json'
for required in [checkpoint/'model.pt',checkpoint/'amanah_config.json',checkpoint/'thresholds.json',cached_model_repository/'reference_store.json',validation_report,acceptance_report]:
    assert required.exists(), f'Missing persisted validated artifact: {required}'
print('Validated cache:',RUN_CACHE)


In [ ]:
# 2) Verify measured reports, then rebuild the package with the latest GitHub runtime code
validation=json.loads(validation_report.read_text(encoding='utf-8'))
acceptance=json.loads(acceptance_report.read_text(encoding='utf-8'))
metrics=validation['evaluation']
assert validation['training_state']['status']=='complete', validation['training_state']
assert metrics['status']=='ok', metrics
assert acceptance['status']=='PASS', acceptance
assert acceptance['training_signature']==RUN_SIGNATURE
print('Macro F1:',metrics['macro_f1'])
print('Critical Drift Recall:',metrics['critical_drift_recall'])
print('False Safe Rate:',metrics['false_safe_rate'])
print('Coverage:',metrics['coverage'])
print('FINAL ACCEPTANCE:',acceptance['status'])

repo_dir=Path('/content/Amanah')
if repo_dir.exists(): shutil.rmtree(repo_dir)
subprocess.run(['git','clone','--depth','1','--branch','amanah-v0.2-precision','https://github.com/Saahar2001/Amanah.git',str(repo_dir)],check=True)
os.chdir(repo_dir)
model_repository=Path('/content/amanah_v02_validated_package')
if model_repository.exists(): shutil.rmtree(model_repository)
subprocess.run([
    sys.executable,'-m','scripts.package_hf_model',
    '--checkpoint',str(checkpoint),
    '--reference-store',str(cached_model_repository/'reference_store.json'),
    '--output',str(model_repository),
],check=True)
service_file=model_repository/'amanah_engine'/'service.py'
service_text=service_file.read_text(encoding='utf-8')
assert 'from data.models import Decision, DriftLabel, Severity' in service_text, service_text.splitlines()[:12]
commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Fresh runtime package built from latest GitHub code:',model_repository)
print('Git commit used for package:',commit)
print('Packaged DriftLabel import: PASS')


In [ ]:
# 3) Full packaged-handler regression gate on CPU
# Fail immediately if this runtime accidentally contains the stale service.py from an older Colab package.
service_file=model_repository/'amanah_engine'/'service.py'
service_text=service_file.read_text(encoding='utf-8')
assert 'from data.models import Decision, DriftLabel, Severity' in service_text, 'STALE PACKAGE: rerun cell 2 to rebuild from latest GitHub code.'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.45,<5','sentencepiece>=0.2,<1','pydantic>=2.8,<3'],check=True)
for key in list(sys.modules):
    if key == 'handler' or key.startswith('amanah_engine') or key.startswith('data.') or key.startswith('training.'):
        sys.modules.pop(key,None)
sys.path.insert(0,str(model_repository.resolve()))
from handler import EndpointHandler
handler=EndpointHandler(str(model_repository))
store=json.loads((model_repository/'reference_store.json').read_text(encoding='utf-8'))
row3528=next(x for x in store['ayahs'] if x['ayah_id']=='35:28')
print('35:28 trusted English references:')
for ref in row3528.get('references_en',[]): print('-',ref)
bad=handler({'inputs':{'source_type':'quran','source_ar':row3528['source_ar'],'candidate_en':'And among people and moving creatures and grazing livestock are likewise various colors. Only Allah fears the knowledgeable among His servants. Indeed, Allah is Exalted in Might and Forgiving.','ayah_id':'35:28'}})
print(json.dumps(bad,ensure_ascii=False,indent=2))
assert bad['decision'] in {'REVIEW','CRITICAL'}, bad
assert any(d.get('label')=='AGENCY_SHIFT' for d in bad.get('drifts',[])), bad

good=handler({'inputs':{'source_type':'quran','source_ar':row3528['source_ar'],'candidate_en':'And among people and moving creatures and grazing livestock are various colors similarly. Only those fear Allah, from among His servants, who have knowledge. Indeed, Allah is Exalted in Might and Forgiving.','ayah_id':'35:28'}})
print('Correct 35:28 candidate:',json.dumps(good,ensure_ascii=False,indent=2))
assert not any(d.get('label')=='AGENCY_SHIFT' for d in good.get('drifts',[])), good

row22=next(x for x in store['ayahs'] if x['ayah_id']=='2:2')
case22=handler({'inputs':{
    'source_type':'quran',
    'source_ar':row22['source_ar'],
    'candidate_en':'This is the Book, beyond doubt, a guidance for the mindful.',
    'ayah_id':'2:2'
}})
print('2:2 semantic-negation control:',json.dumps(case22,ensure_ascii=False,indent=2))
assert not any(d.get('label')=='NEGATION_FLIP' for d in case22.get('drifts',[])), case22
assert case22['decision']!='CRITICAL', case22

# Ensure the true polarity-change class is still caught and was not broadly suppressed.
real_flip=handler({'inputs':{
    'source_type':'quran',
    'source_ar':row22['source_ar'],
    'candidate_en':'This is not the Book, a guidance for the mindful.',
    'ayah_id':'2:2'
}})
print('2:2 true-negation control:',json.dumps(real_flip,ensure_ascii=False,indent=2))
assert any(d.get('label')=='NEGATION_FLIP' for d in real_flip.get('drifts',[])), real_flip
assert real_flip['decision'] in {'REVIEW','CRITICAL'}, real_flip

print('CPU FULL-HANDLER RELEASE GATES: PASS')


In [ ]:
# 4) Upload the FRESH validated package to the EXISTING Hugging Face model repo; no endpoint is created
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.35,<2'],check=True)
from huggingface_hub import HfApi
hf_token=userdata.get('HF_TOKEN')
if not hf_token: raise RuntimeError('Add your existing Hugging Face WRITE token to Colab Secrets as HF_TOKEN. Do not buy or create anything new.')
api=HfApi(token=hf_token)
identity=api.whoami(); namespace=identity.get('name'); assert namespace
MODEL_REPO=f'{namespace}/semantic-integrity-v0-1'
api.repo_info(repo_id=MODEL_REPO,repo_type='model')
try:
    api.create_branch(repo_id=MODEL_REPO,branch='v0.2-validated',repo_type='model',exist_ok=True,token=hf_token)
except TypeError:
    try: api.create_branch(repo_id=MODEL_REPO,branch='v0.2-validated',repo_type='model',token=hf_token)
    except Exception: pass
api.upload_folder(repo_id=MODEL_REPO,repo_type='model',folder_path=str(model_repository),revision='v0.2-validated',token=hf_token,commit_message='Publish patched AMANAH v0.2 validated runtime')
api.upload_file(path_or_fileobj=str(validation_report),path_in_repo='VALIDATION_REPORT.json',repo_id=MODEL_REPO,repo_type='model',revision='v0.2-validated',token=hf_token,commit_message='Add measured v0.2 validation report')
api.upload_file(path_or_fileobj=str(acceptance_report),path_in_repo='FINAL_ACCEPTANCE_REPORT.json',repo_id=MODEL_REPO,repo_type='model',revision='v0.2-validated',token=hf_token,commit_message='Add v0.2 acceptance report')
print('HF VALIDATED UPLOAD: PASS')
print('Repository:',f'https://huggingface.co/{MODEL_REPO}/tree/v0.2-validated')
print('No Hugging Face endpoint, subscription, payment method, or credits were created by this notebook.')


## Next step (manual, existing endpoint only)
After the upload succeeds, open your **existing** Hugging Face Inference Endpoint and update its repository revision to `v0.2-validated`. Do not create a second endpoint. Then run the existing endpoint smoke/live gate notebook or the website's 35:28 test before treating the deployment as complete.
